In [1]:
from datetime import datetime

import gymnasium as gym
import torch
from torch.utils.tensorboard import SummaryWriter

from src.networks.dqn import DQN
from src.networks.mlp import MLPBox
from src.training import (
    RolloutBatch,
    assemble_dataset,
    compute_advantages,
    evaluate_action_continuous,
    get_action_continuous,
    rollout_trajectory,
    train,
    update_critic,
    update_policy,
)
from src.utils import save_checkpoint



In [2]:
# ENV_NAME = "MountainCarContinuous-v0"
ENV_NAME = "LunarLander-v3"
NUM_ENVS = 8

In [3]:
LR = 3e-4
NUM_EPISODES = 2000
NUM_STEPS=1024
K_EPOCH = 4
GAMMA = .99
LAMBDA = 0.9

In [4]:
def make_env():
    env = gym.make(
        id=ENV_NAME,
        continuous=True,
        gravity=-10.0,
        enable_wind=False,
        wind_power=15.0,
        turbulence_power=1.5,
        render_mode = None
    )
    env = gym.wrappers.RecordEpisodeStatistics(env)
    return env

In [5]:
envs = gym.vector.SyncVectorEnv([make_env for _ in range(NUM_ENVS)])

In [6]:
N_OBS, N_ACT = envs.envs[0].observation_space.shape[0],envs.envs[0].action_space.shape[0]
N_OBS, N_ACT

(8, 2)

In [7]:
policy = MLPBox(N_OBS, N_ACT)
critic = DQN(N_OBS, 1, 20)

optimizer_policy = torch.optim.Adam(policy.net.parameters(), LR)
optimizer_critic = torch.optim.Adam(critic.net.parameters(), LR)

total_updates = NUM_EPISODES * NUM_STEPS * K_EPOCH

def lr_lambda(step):
    return max(1.0 - (step / total_updates), 1.5e-3) 

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer_policy, lr_lambda)

In [8]:
def action_fn(state: torch.Tensor):
    a, p = get_action_continuous(policy, state)
    return torch.clamp(a, -1.0, 1.0), a, p

def reward_bonus_fn(p,n):
    return 0

def calc_advantage_fn(rollout: RolloutBatch, gamma, lam):
    return compute_advantages(critic, rollout, gamma, lam)

def upd_critic(batch):
    loss = update_critic(critic, batch, GAMMA)
    optimizer_critic.zero_grad()
    loss.backward()
    optimizer_critic.step()

def eval_action(observations: torch.Tensor, actions: torch.Tensor):
    return evaluate_action_continuous(policy, observations, actions)

def upd_policy(batch):
    loss = update_policy(batch, eval_action)
    optimizer_policy.zero_grad()
    loss.backward()
    optimizer_policy.step()
    scheduler.step()

In [9]:
best_reward_tracker = [float("-inf")]

In [10]:
run_name = f"{ENV_NAME}_ppo_ne{NUM_EPISODES}_{datetime.now()}"
writer = SummaryWriter(f"./logs/{run_name}")

In [ ]:
for e in range(NUM_EPISODES):

    true_ep = NUM_EPISODES * 2 + e + 1
    traj, episode_returns, ep_len = rollout_trajectory(envs, action_fn, reward_bonus_fn, NUM_STEPS)

    advantages, values = calc_advantage_fn(traj, GAMMA, LAMBDA)
    
    ds, ds_len = assemble_dataset(traj, advantages, values)

    for _ in range(K_EPOCH):
        train(ds, ds_len, upd_critic, upd_policy)

    if len(episode_returns):
        sum_reward = float(sum(episode_returns) / len(episode_returns))
        sum_ep_len = float(sum(ep_len) / len(episode_returns))
    else:
        sum_reward = 0
        sum_ep_len = 0
    writer.add_scalar("train/reward", sum_reward, true_ep)
    writer.add_scalar("train/len", sum_ep_len, true_ep)

    state_to_save = {
        "policy_state_dict": policy.state_dict(),
        "optimizer_policy_state_dict": optimizer_policy.state_dict(),
        "critic_state_dict": critic.state_dict(),
        "optimizer_critic_state_dict": optimizer_critic.state_dict(),
    }

    save_checkpoint(
        state_dict=state_to_save,
        epoch=true_ep,
        reward=sum_reward,
        checkpoint_dir="./data/checkpoints",
        ttl_epochs=NUM_EPISODES // 10,
        best_reward_tracker=best_reward_tracker,
    )

In [20]:
from src.utils import run_tests

num_tests = 100

test_env = make_env()

test_policy = MLPBox(N_OBS, N_ACT)
checkpoint = torch.load("./data/checkpoints/checkpoint_epoch_3600.pt", weights_only=False)
test_policy.load_state_dict(checkpoint["policy_state_dict"])
test_policy.eval()

class Mock:
    def __init__(self, net):
        self.net = net

    def act(self, x):
        obs = torch.Tensor(x)
        a, p = get_action_continuous(self.net, obs)
        return torch.clamp(a, -1.0, 1.0).numpy()


test_agent = Mock(test_policy)
avg, _min, _max, _ = run_tests(test_agent, test_env, writer, num_tests)
print("avg: ", avg, _min, _max)

avg:  179.77701577201944 -5.609914215058851 294.3737562003422


In [ ]:
policy_out = MLPBox(N_OBS, N_ACT)
checkpoint = torch.load("./data/checkpoints/checkpoint_epoch_3600.pt", weights_only=False)
policy_out.load_state_dict(checkpoint["policy_state_dict"])
policy_out.eval()

dummy_input = torch.randn(1, N_OBS)
torch.onnx.export(
    policy_out,
    dummy_input,
    f"./data/latest.onnx",
    input_names=["obs"],
    output_names=["action_probs"],
    dynamic_axes={"obs": {0: "batch"}, "action_probs": {0: "batch"}},
    external_data=False,
)
print(f"./data/{run_name}.onnx")